# Laya vs JEV Book Classification Benchmark

Upload a `books.json` dataset, select how many books to process, classify books in batches of 100, and compare Laya against JEV.

Only `title` and `description` are sent to the classifiers.

In [ ]:
# Install dependencies
!pip -q install laya typesafe-sdk pandas matplotlib tqdm


In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# If torch is not installed in envronment, run the following command to install it
# !pip -q install laya typesafe-sdk pandas matplotlib tqdm torch

In [ ]:
import os
import json
import time
import statistics

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from google.colab import files, userdata
from laya import Router
from typesafe_sdk import TypeSafeClient

BATCH_SIZE = 100
MAX_BOOKS = 1000  # Set to None to process the whole uploaded file.
LAYA_MODEL = "typed-decisions"  # Set to None for automatic Laya routing.
JEV_MODEL = "jev-latest"

try:
    JEV_API_KEY = userdata.get("JEV_API_KEY")
except Exception:
    JEV_API_KEY = None

if not JEV_API_KEY:
    raise RuntimeError(
        "JEV_API_KEY was not found. Add JEV_API_KEY in Google Colab "
        "Secrets and enable notebook access."
    )

os.environ["TYPESAFE_API_KEY"] = JEV_API_KEY

print("JEV_API_KEY loaded from Colab Secret.")

In [ ]:
BOOK_CATEGORIES = {
    "fiction": "Novels and general literary fiction focused primarily on an invented story or characters.",
    "romance": "Stories primarily centered on romantic relationships, love, attraction, or romantic relationships.",
    "mystery_thriller": "Mystery, crime, detective, suspense, thriller, investigation, or psychological suspense stories.",
    "fantasy": "Stories involving magic, supernatural worlds, mythical creatures, or fantastical elements.",
    "science_fiction": "Stories primarily involving futuristic technology, space, scientific concepts, speculative technology, or imagined scientific societies.",
    "historical": "Books primarily centered on historical periods, historical events, or stories substantially set in the past.",
    "biography_memoir": "Biographies, autobiographies, memoirs, or books primarily about a real person's life.",
    "philosophy_religion": "Books primarily about philosophy, ethics, religion, theology, metaphysics, or philosophical ideas.",
    "history_politics": "Nonfiction primarily focused on history, politics, governments, wars, civilizations, or political events.",
    "travel_culture": "Books primarily about travel, destinations, geography, places, cultures, or travel experiences.",
    "children_young_adult": "Books primarily written for children, middle-grade readers, or young adult readers.",
    "other_nonfiction": "Nonfiction that does not clearly fit the other categories."
}

CLASSIFICATION_QUESTION = {
    "category": {
        "type": "choice",
        "instructions": (
            "Which category best describes this book based only on its title "
            "and description? Select the single most appropriate primary category."
        ),
        "criteria": BOOK_CATEGORIES,
    }
}

print(f"Categories: {len(BOOK_CATEGORIES)}")
for category in BOOK_CATEGORIES:
    print(f"  - {category}")

In [ ]:
# Initialize Laya
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Laya on {device}...")

laya_router = Router(
    preload=True,
    device=device,
)

print(f"Laya loaded on {device}.")

In [ ]:
# Upload books.json
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded.")

uploaded_filename = next(iter(uploaded.keys()))

if not uploaded_filename.lower().endswith(".json"):
    raise ValueError("Please upload a .json file.")

with open(uploaded_filename, "r", encoding="utf-8") as f:
    books = json.load(f)

if not isinstance(books, list):
    raise ValueError("Expected books.json to contain a JSON array.")

print(f"Loaded {len(books):,} records.")

In [ ]:
# Normalize dataset. Only title and description are retained.
normalized_books = []

for index, book in enumerate(books):
    title = str(book.get("title", "")).strip()
    description = str(book.get("description", "")).strip()

    if not title and not description:
        continue

    normalized_books.append({
        "id": index,
        "title": title,
        "description": description,
    })

if MAX_BOOKS is not None:
    normalized_books = normalized_books[:MAX_BOOKS]

print(f"Books selected: {len(normalized_books):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of batches: {(len(normalized_books) + BATCH_SIZE - 1) // BATCH_SIZE}")

In [ ]:
def build_state(book):
    return {
        "title": book["title"],
        "description": book["description"],
    }


def extract_laya_result(response):
    answer = response["answers"]["category"]
    return {
        "category": answer.get("choice"),
        "confidence": answer.get("confidence"),
        "routing_model": response.get("routing", {}).get("model"),
    }


def extract_jev_result(response):
    answer = response.answers["category"]
    return {
        "category": answer.choice,
        "confidence": answer.confidence,
    }

In [ ]:
# Run benchmark
jev_client = TypeSafeClient(model=JEV_MODEL)

results = []
laya_latencies = []
jev_latencies = []
laya_errors = 0
jev_errors = 0
batch_stats = []

def classify_laya(book):
    start = time.perf_counter()
    try:
        response = laya_router.predict(
            build_state(book),
            CLASSIFICATION_QUESTION,
            **({"model": LAYA_MODEL} if LAYA_MODEL else {})
        )
        elapsed = time.perf_counter() - start
        return extract_laya_result(response), elapsed, None
    except Exception as e:
        elapsed = time.perf_counter() - start
        return None, elapsed, repr(e)


def classify_jev(book):
    start = time.perf_counter()
    try:
        response = jev_client.system_one(
            state=build_state(book),
            questions=CLASSIFICATION_QUESTION,
        )
        elapsed = time.perf_counter() - start
        return extract_jev_result(response), elapsed, None
    except Exception as e:
        elapsed = time.perf_counter() - start
        return None, elapsed, repr(e)


total_start = time.perf_counter()

with jev_client:
    for batch_start in range(0, len(normalized_books), BATCH_SIZE):
        batch = normalized_books[batch_start:batch_start + BATCH_SIZE]
        batch_number = batch_start // BATCH_SIZE + 1
        batch_start_time = time.perf_counter()

        for book in tqdm(batch, desc=f"Batch {batch_number}", leave=False):
            laya_result, laya_time, laya_error = classify_laya(book)
            jev_result, jev_time, jev_error = classify_jev(book)

            if laya_error:
                laya_errors += 1
            else:
                laya_latencies.append(laya_time)

            if jev_error:
                jev_errors += 1
            else:
                jev_latencies.append(jev_time)

            results.append({
                "id": book["id"],
                "title": book["title"],
                "description": book["description"],
                "laya_category": laya_result["category"] if laya_result else None,
                "laya_confidence": laya_result["confidence"] if laya_result else None,
                "laya_routing_model": laya_result["routing_model"] if laya_result else None,
                "laya_latency_ms": laya_time * 1000,
                "laya_error": laya_error,
                "jev_category": jev_result["category"] if jev_result else None,
                "jev_confidence": jev_result["confidence"] if jev_result else None,
                "jev_latency_ms": jev_time * 1000,
                "jev_error": jev_error,
                "agreement": (
                    laya_result is not None
                    and jev_result is not None
                    and laya_result["category"] == jev_result["category"]
                ),
            })

        batch_elapsed = time.perf_counter() - batch_start_time
        batch_stats.append({
            "batch": batch_number,
            "books": len(batch),
            "elapsed_seconds": batch_elapsed,
            "throughput_books_per_second": len(batch) / batch_elapsed,
        })

total_elapsed = time.perf_counter() - total_start
print("Benchmark completed.")

In [ ]:
df = pd.DataFrame(results)
display(df[[
    "title", "laya_category", "laya_confidence",
    "jev_category", "jev_confidence", "agreement"
]].head(20))

In [ ]:
def percentile(values, p):
    if not values:
        return float("nan")
    return float(pd.Series(values).quantile(p / 100))


def benchmark_stats(values, total_books, errors):
    total = sum(values)
    return {
        "total_latency_s": total,
        "mean_ms": statistics.mean(values) * 1000 if values else float("nan"),
        "median_ms": statistics.median(values) * 1000 if values else float("nan"),
        "p95_ms": percentile(values, 95) * 1000,
        "p99_ms": percentile(values, 99) * 1000,
        "throughput_books_s": len(values) / total if total > 0 else float("nan"),
        "errors": errors,
        "error_rate_%": errors / total_books * 100 if total_books else float("nan"),
    }

laya_stats = benchmark_stats(laya_latencies, len(normalized_books), laya_errors)
jev_stats = benchmark_stats(jev_latencies, len(normalized_books), jev_errors)

benchmark_df = pd.DataFrame([laya_stats, jev_stats], index=["Laya", "Jev"])
display(benchmark_df.round(3))

In [ ]:
valid = df[df["laya_category"].notna() & df["jev_category"].notna()]
agreement_rate = valid["agreement"].mean() * 100 if len(valid) else 0

print(f"Agreement: {agreement_rate:.2f}%")
print(f"Compared predictions: {len(valid):,}")

display(valid[~valid["agreement"]][[
    "title", "laya_category", "laya_confidence",
    "jev_category", "jev_confidence"
]].head(20))

In [ ]:
# Latency summary
plt.figure(figsize=(10, 6))
plt.bar(["Laya", "Jev"], [laya_stats["median_ms"], jev_stats["median_ms"]])
plt.ylabel("Median latency (ms)")
plt.title("Median Classification Latency")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# P95 latency
plt.figure(figsize=(10, 6))
plt.bar(["Laya", "Jev"], [laya_stats["p95_ms"], jev_stats["p95_ms"]])
plt.ylabel("Latency (ms)")
plt.title("P95 Classification Latency")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Throughput
plt.figure(figsize=(10, 6))
plt.bar(["Laya", "Jev"], [laya_stats["throughput_books_s"], jev_stats["throughput_books_s"]])
plt.ylabel("Books / second")
plt.title("Classification Throughput")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Latency distribution
plt.figure(figsize=(12, 6))
plt.hist([x * 1000 for x in laya_latencies], bins=40, alpha=0.6, label="Laya")
plt.hist([x * 1000 for x in jev_latencies], bins=40, alpha=0.6, label="Jev")
plt.xlabel("Latency (ms)")
plt.ylabel("Number of predictions")
plt.title("Latency Distribution")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Classification distribution
laya_counts = df["laya_category"].value_counts()
jev_counts = df["jev_category"].value_counts()
categories = sorted(set(laya_counts.index) | set(jev_counts.index))
x = range(len(categories))
width = 0.4

plt.figure(figsize=(15, 7))
plt.bar([i - width / 2 for i in x], [laya_counts.get(c, 0) for c in categories], width=width, label="Laya")
plt.bar([i + width / 2 for i in x], [jev_counts.get(c, 0) for c in categories], width=width, label="Jev")
plt.xticks(list(x), categories, rotation=45, ha="right")
plt.ylabel("Number of books")
plt.title("Classification Distribution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Batch throughput
batch_df = pd.DataFrame(batch_stats)
plt.figure(figsize=(12, 6))
plt.plot(batch_df["batch"], batch_df["throughput_books_per_second"], marker="o")
plt.xlabel("Batch")
plt.ylabel("Books / second")
plt.title("Processing Throughput per Batch")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Confidence comparison
confidence_df = df[df["laya_confidence"].notna() & df["jev_confidence"].notna()]
plt.figure(figsize=(8, 8))
plt.scatter(confidence_df["laya_confidence"], confidence_df["jev_confidence"], alpha=0.35)
plt.xlabel("Laya confidence")
plt.ylabel("Jev confidence")
plt.title("Laya vs Jev Confidence")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Final report
print("=" * 70)
print("BOOK CLASSIFICATION BENCHMARK")
print("=" * 70)
print(f"\nBooks processed : {len(df):,}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Total wall time : {total_elapsed:.2f}s")

for name, stats in [("Laya", laya_stats), ("Jev", jev_stats)]:
    print(f"\n{name}")
    print("-" * 30)
    print(f"Mean latency    : {stats['mean_ms']:.2f} ms")
    print(f"Median latency  : {stats['median_ms']:.2f} ms")
    print(f"P95 latency     : {stats['p95_ms']:.2f} ms")
    print(f"P99 latency     : {stats['p99_ms']:.2f} ms")
    print(f"Throughput      : {stats['throughput_books_s']:.2f} books/s")
    print(f"Errors          : {stats['errors']}")

print(f"\nLaya/Jev agreement: {agreement_rate:.2f}%")

In [ ]:
# Export benchmark results
output_path = "books_laya_jev_benchmark.json"

export_data = {
    "config": {
        "books_requested": MAX_BOOKS,
        "books_processed": len(df),
        "batch_size": BATCH_SIZE,
        "laya_model": LAYA_MODEL,
        "jev_model": JEV_MODEL,
        "categories": BOOK_CATEGORIES,
    },
    "benchmark": {
        "total_wall_time_seconds": total_elapsed,
        "laya": laya_stats,
        "jev": jev_stats,
        "agreement_percent": agreement_rate,
    },
    "books": df.to_dict(orient="records"),
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print(f"Saved: {output_path}")
files.download(output_path)